# WIDS: Deauth, Disassociation and Reassociation Detection (AWID3)

Prototype Wi-Fi intrusion detector using a subset of AWID3. This notebook analyses three management-frame attacks: deauthentication, disassociation and (re)association.

## Deauthentication flood detection (AWID3)

Goal: detect deauthentication floods from 1-second windows of 802.11 frames, using ML, then raise alerts on a replayed stream. The attack only appears in the later parts of the Deauth folder, so the parts are read one at a time (batching) and reduced to small per-second summaries.

In [1]:
import os, re
import numpy as np
import pandas as pd

root = '/kaggle/input/datasets/suumia/awid3-dataset/CSV'
folder = os.path.join(root, "1.Deauth")

def part_no(f):
    return int(re.findall(r"\d+", f)[-1])

deauth_parts = sorted(os.listdir(folder), key=part_no)
print(len(deauth_parts), "parts")

33 parts


## 1. Label check on one attack part

`Deauth_22` is loaded in full to check that the labels agree with the real frame type and subtype (type 0 = management, subtype 12 = deauthentication, subtype 10 = disassociation).

In [2]:
p = pd.read_csv(os.path.join(folder, "Deauth_22.csv"), low_memory=False)
print(pd.crosstab(p["Label"], [p["wlan.fc.type"], p["wlan.fc.subtype"]]))

wlan.fc.type     0                                         ...    1        \
wlan.fc.subtype 0  1   2   3    4   5    8   10  11    12  ...   8     9    
Label                                                      ...              
Deauth           0  0   0   0    0   0    0  14   0  5187  ...    0     0   
Normal           4  3  18  19  128  53  214  22  44     1  ...  462  5227   

wlan.fc.type                          2                    
wlan.fc.subtype     11    12    13   0   4      8  9   12  
Label                                                      
Deauth               0     0     0    0   0      0  0   0  
Normal           11560  6840  4487  112  35  14129  3  83  

[2 rows x 23 columns]


### Findings
- 5,187 of the 5,201 `Deauth` frames are type 0, subtype 12. The other 14 are subtype 10 (disassociation).
- Only 1 `Normal` frame is a deauth frame, so a deauth count should separate the classes very well. High scores are expected.
- 22 `Normal` frames are disassociation frames, so disassociation will be harder to detect.

## 2. Who sends the attack frames?

The attack frames are inspected for sender, receiver, BSSID and reason code, because this decides what the alert should log.

In [3]:
atk = p[p["Label"] == "Deauth"]
for c in ["wlan.sa", "wlan.da", "wlan.bssid", "wlan.fixed.reason_code"]:
    print(atk[c].value_counts().head(5), "\n")

wlan.sa
0c:9d:92:54:fe:34    2643
b4:ce:40:c8:33:81    1332
88:66:a5:55:a2:d4    1030
94:e9:79:82:c5:77     196
Name: count, dtype: int64 

wlan.da
0c:9d:92:54:fe:34    2558
b4:ce:40:c8:33:81    1497
88:66:a5:55:a2:d4     922
94:e9:79:82:c5:77     210
00:c0:ca:a8:26:3e      14
Name: count, dtype: int64 

wlan.bssid
0c:9d:92:54:fe:34    5201
Name: count, dtype: int64 

wlan.fixed.reason_code
7.0    4332
1.0     671
6.0     196
3.0       2
Name: count, dtype: int64 



### 🔍 Protocol-Level Analysis of Deauthentication Traffic (`Deauth_22.csv`)

To verify that the dataset labels correspond to genuine wireless attacks rather than anomalies, we inspected key IEEE 802.11 management frame fields (`wlan.sa`, `wlan.da`, `wlan.bssid`, and `wlan.fixed.reason_code`) for rows labeled as `Deauth`.

#### 1. Transmitter & Receiver Analysis (`wlan.sa`, `wlan.da`, `wlan.bssid`)
* **BSSID Consistency:** Every single attack frame points back to a single BSSID (`0c:9d:92:54:fe:34`), indicating that the attacker is impersonating or targeting a specific Access Point.
* **Source/Destination Pairs:** The primary sender (`wlan.sa`) and receiver (`wlan.da`) addresses match the flood pattern of an automated injection tool (e.g., `aireplay-ng`), where tear-down commands are blasted toward connected clients.

#### 2. Reason Code Breakdown (`wlan.fixed.reason_code`)
In the 802.11 standard, deauthentication frames carry reason codes explaining why the link is severed. The attack traffic is heavily dominated by programmatic codes:
* **Reason Code 7 (`4,332` instances):** *"Class 3 frame received from nonassociated station."* This is the hallmark signature of automated deauth flood tools, as frames are sent blindly to drop active client associations.
* **Reason Code 1 (`671` instances):** *"Unspecified reason."* A generic code utilized by custom spoofing scripts.
* **Reason Code 6 (`196` instances):** *"Class 2 frame received from nonauthenticated station."*

#### 💡 Takeaway for Feature Engineering
These fields confirm that features tracking **reason code frequencies**, **per-MAC packet velocity**, and **BSSID association anomalies** will serve as highly discriminative features for our Decision Tree and Random Forest classifiers.

## 3. Build 1-second windows in batches

Each part (about 50,000 frames) is read one at a time with only the needed columns. For each second, the frames are counted by kind. Seconds with no deauth frames are still included (their count is 0), because every second that has any frame gets a row. The deauth frames themselves are kept separately (they are few) so that alerts can report the BSSID and claimed senders.

A second can span two parts, so the per-part summaries are added together by second afterwards.

In [4]:
USE = ["frame.time_epoch", "wlan.fc.type", "wlan.fc.subtype",
       "wlan.sa", "wlan.da", "wlan.bssid", "Label"]

def read_part(path):
    c = pd.read_csv(path, usecols=USE, low_memory=False)
    c["sec"] = np.floor(c["frame.time_epoch"].astype(float)).astype("int64")
    is_mgmt   = c["wlan.fc.type"] == 0
    c["is_mgmt"]   = is_mgmt
    c["is_deauth"] = is_mgmt & (c["wlan.fc.subtype"] == 12)
    c["is_disas"]  = is_mgmt & (c["wlan.fc.subtype"] == 10)
    c["is_attack"] = c["Label"] != "Normal"

    win = c.groupby("sec").agg(
        n_frames=("sec", "size"), n_mgmt=("is_mgmt", "sum"),
        n_deauth=("is_deauth", "sum"), n_disas=("is_disas", "sum"),
        n_attack=("is_attack", "sum")).reset_index()
    frames = c.loc[c["is_deauth"], ["sec", "wlan.sa", "wlan.da", "wlan.bssid"]]
    return win, frames

wins, deauth_frames = [], []
for f in deauth_parts:
    w, fr = read_part(os.path.join(folder, f))
    wins.append(w); deauth_frames.append(fr)

w = pd.concat(wins).groupby("sec", as_index=False).sum().sort_values("sec")
deauth_frames = pd.concat(deauth_frames)

w["frac_deauth"] = w["n_deauth"] / w["n_mgmt"].replace(0, np.nan)
w["frac_deauth"] = w["frac_deauth"].fillna(0)
w["label"] = (w["n_attack"] > 0).astype(int)      # 1 = attack window

print(w.shape)
print(w["label"].value_counts())
w.head()

(604, 8)
label
0    424
1    180
Name: count, dtype: int64


,sec,n_frames,n_mgmt,n_deauth,n_disas,n_attack,frac_deauth,label
0,1608306178,10,6,0,0,0,0.0,0
1,1608306179,35,29,0,0,0,0.0,0
2,1608306180,15,10,0,0,0,0.0,0
3,1608306181,16,10,0,0,0,0.0,0
4,1608306182,15,10,0,0,0,0.0,0


### Time-Window Feature Aggregation (1-Second Bins)

To effectively detect burst-based attacks like deauthentication floods, analyzing individual packets in isolation can be noisy and computationally inefficient. Instead, we transformed the packet-level data into **1-second aggregated time windows**.

#### 1. Processing & Feature Logic
Using the selected columns (`frame.time_epoch`, `wlan.fc.type`, `wlan.fc.subtype`, `wlan.sa`, `wlan.da`, `wlan.bssid`, `Label`), each packet was processed as follows:
* **Epoch Second Mapping:** Converted packet timestamps into discrete second blocks (`sec`).
* **Frame Filtering:** Isolated Wi-Fi management frames (`wlan.fc.type == 0`) and targeted subtypes, specifically identifying deauthentication frames (`subtype == 12`).
* **Window Grouping:** Grouped data by second to calculate core traffic velocity metrics: total frames (`n_frames`), management frames (`n_mgmt`), deauthentication counts (`n_deauth`), and general attack flags (`n_attack`).

#### 2. Window-Level Dataset Summary
Aggregating the selected attack chunks into 1-second intervals yielded a well-proportioned time-series structure:
* **Total Time Windows (`w.shape`):** **604 seconds** (~10 minutes of traffic duration).
* **Class Distribution (`w["label"].value_counts()`):**
  * **Normal Windows (`0`):** `424` seconds
  * **Attack Windows (`1`):** `180` seconds

#### 💡 Takeaway
This windowing approach transforms high-frequency packet captures into a manageable time-series format (~70% normal vs. ~30% attack windows). It smooths out packet-level variance while highlighting the sharp velocity spikes characteristic of a deauth flood, making it ideal for supervised classification.

In [5]:
print(w.groupby("label")[["n_deauth", "n_disas", "n_frames", "frac_deauth"]].describe().T)

label                         0             1
n_deauth    count    424.000000    180.000000
            mean       0.000000    216.088889
            std        0.000000     55.723609
            min        0.000000     45.000000
            25%        0.000000    176.500000
            50%        0.000000    227.000000
            75%        0.000000    261.500000
            max        0.000000    310.000000
n_disas     count    424.000000    180.000000
            mean       0.007075      0.638889
            std        0.145693      1.348551
            min        0.000000      0.000000
            25%        0.000000      0.000000
            50%        0.000000      0.000000
            75%        0.000000      1.000000
            max        3.000000      7.000000
n_frames    count    424.000000    180.000000
            mean    2561.162736   3002.977778
            std     2799.626644   2073.525247
            min       10.000000    497.000000
            25%       57.000000   